In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

from config import DATASETS, DEDUP_KEYS, bronze_table, silver_table

In [ ]:
for dataset in DATASETS:
    bronze_df = spark.table(bronze_table(dataset))

    window = Window.partitionBy(*DEDUP_KEYS[dataset]).orderBy(F.col("_ingested_at").desc())

    silver_df = (
        bronze_df
        .withColumn("_row_number", F.row_number().over(window))
        .filter(F.col("_row_number") == 1)
        .drop("_row_number", "_source_file", "_ingested_at")
    )

    silver_df.write.format("delta").mode("overwrite").saveAsTable(silver_table(dataset))

    print(f"{dataset}: {silver_df.count()} linhas gravadas em {silver_table(dataset)}")